# REVERSE 직접 돌려보기: VLM 환각 줄이기 (Qwen2.5-VL-3B)

논문: [Generate, but Verify (NeurIPS 2025)](https://arxiv.org/abs/2504.13169) · 코드: [tsunghan-wu/reverse_vlm](https://github.com/tsunghan-wu/reverse_vlm)

이 노트북은 세 단계로 진행합니다.

1. **일반 사진**: 원래 모델(Qwen2.5-VL-3B-Instruct)과 REVERSE 모델의 설명을 비교합니다.
2. **구절별 확신도 보기**: REVERSE가 각 구절에 붙이는 "자신 없음(`</UN>`)" 확률을 표로 확인합니다.
3. **안저 영상**: 공개 안저 사진을 넣어서 REVERSE가 의료 영상에서도 자신 없는 부분을 잘 표시하는지 봅니다.

**시작 전**: 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택하세요. 학습은 하지 않고, 공개된 모델로 추론만 합니다.

## 0. 설치

In [ ]:
!pip -q install "transformers>=4.49" accelerate qwen-vl-utils pandas
!git clone -q https://github.com/tsunghan-wu/reverse_vlm

In [ ]:
import gc, importlib.util, io, types
import requests, torch, pandas as pd
from PIL import Image
from IPython.display import display
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# Load REVERSE's inference logic straight from the file.
# (Importing the qwenvl package would pull in training-only dependencies.)
spec = importlib.util.spec_from_file_location("reverse_infer", "reverse_vlm/qwenvl/eval/inference_utils.py")
reverse_infer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reverse_infer)

DEVICE = "cuda"
DTYPE = torch.float16  # T4 GPUs do not support bfloat16 well
print(torch.cuda.get_device_name(0))

## 1. 공통 함수

- `load_model`: 모델과 프로세서를 불러옵니다.
- `build_inputs`: 이미지와 질문을 모델 입력으로 바꿉니다.
- `plain_generate`: 일반 생성입니다. 원래 모델에 사용합니다.
- `reverse_generate`: REVERSE의 되돌아가서 다시 쓰기(retrospective resampling)입니다.
- `phrase_confidence`: 한 번 생성한 뒤, 구절마다 `</UN>`(자신 없음) 확률을 계산합니다.

In [ ]:
def load_model(model_id):
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, attn_implementation="sdpa", device_map="auto"
    ).eval()
    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor


def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def load_image(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if src.startswith("http"):
        return Image.open(io.BytesIO(requests.get(src, timeout=30).content)).convert("RGB")
    return Image.open(src).convert("RGB")


def build_inputs(processor, image, question):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], padding=True, return_tensors="pt").to(DEVICE)


@torch.inference_mode()
def plain_generate(model, processor, image, question, max_new_tokens=300):
    inputs = build_inputs(processor, image, question)
    out = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


def reverse_args(un_threshold=0.01, max_new_tokens=300, verbose=False):
    # Settings follow the paper's Qwen2.5-VL setup (tau = 0.01).
    return types.SimpleNamespace(
        temperature=0.0, top_p=None, num_beams=1, max_new_tokens=max_new_tokens,
        un_threshold=un_threshold, temperature_step=0.05,
        max_total_generation_attempts=50, max_local_generation_attempts=10,
        verbose=verbose, open_ended_qa=False,
    )


@torch.inference_mode()
def reverse_generate(model, processor, image, question, **kw):
    inputs = build_inputs(processor, image, question)
    text, history, ok = reverse_infer.retrospective_resampling_generate(
        model, processor.tokenizer, inputs, reverse_args(**kw)
    )
    flagged = [h for _, h in history if h not in (None, "successful correction")]
    return text, flagged, ok


@torch.inference_mode()
def phrase_confidence(model, processor, image, question, max_new_tokens=300):
    # One greedy pass; for every <SPAN> ... </CN|UN> phrase, report P(</UN>) at the closing position.
    tok = processor.tokenizer
    inputs = build_inputs(processor, image, question)
    out = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens,
                         return_dict_in_generate=True, output_logits=True)
    seq = out.sequences[0, inputs.input_ids.shape[1]:]
    probs = torch.softmax(torch.stack(out.logits, dim=1)[0].float(), dim=-1)
    span_id, un_id, cn_id = (tok.convert_tokens_to_ids(t) for t in ["<SPAN>", "</UN>", "</CN>"])
    rows, start = [], None
    for i, t in enumerate(seq.tolist()):
        if t == span_id:
            start = i
        elif t in (un_id, cn_id) and start is not None:
            rows.append({
                "phrase": tok.decode(seq[start + 1:i], skip_special_tokens=True).strip(),
                "P(</UN>)": round(probs[i, un_id].item(), 4),
                "tag": "UN (unsure)" if t == un_id else "CN (confident)",
            })
            start = None
    return pd.DataFrame(rows)

## 2. 일반 사진으로 비교하기

COCO 데이터셋의 유명한 고양이 사진(소파 위 고양이 두 마리와 리모컨 두 개)으로 시작합니다. 원래 모델과 REVERSE 모델이 사진에 없는 물건을 지어내는지 비교해 보세요.

GPU 메모리를 아끼려고 두 모델을 차례로 불러옵니다.

In [ ]:
IMAGES = {
    "coco_cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "coco_kitchen": "http://images.cocodataset.org/val2017/000000397133.jpg",
}
QUESTION = "Describe this image in detail."

for name, url in IMAGES.items():
    print(name); display(load_image(url).resize((320, 240)))

In [ ]:
# 2-1. Base model (official Qwen2.5-VL-3B-Instruct)
base_model, base_proc = load_model("Qwen/Qwen2.5-VL-3B-Instruct")
results = []
for name, url in IMAGES.items():
    ans = plain_generate(base_model, base_proc, load_image(url), QUESTION)
    results.append({"image": name, "model": "base", "answer": ans})
    print(f"[{name}] BASE\n{ans}\n")
del base_model, base_proc
clear_gpu()

In [ ]:
# 2-2. REVERSE model
rev_model, rev_proc = load_model("tsunghanwu/reverse_qwen25_vl")
for name, url in IMAGES.items():
    ans, flagged, ok = reverse_generate(rev_model, rev_proc, load_image(url), QUESTION)
    results.append({"image": name, "model": "reverse", "answer": ans,
                    "flagged_phrases": "; ".join(flagged), "fully_corrected": ok})
    print(f"[{name}] REVERSE\n{ans}\n  flagged: {flagged}\n")

**확인할 것**: 원래 모델의 설명에 사진에 없는 물건이 있나요? REVERSE가 그런 구절을 `flagged`로 잡아서 고쳤나요?

참고로 논문의 공정한 비교 대상은 같은 10만 개 데이터로 학습한 기본 모델이에요. 여기서는 공식 Instruct 모델과 비교하니 수치보다 경향을 보세요.

## 3. 구절별 확신도 보기

REVERSE는 핵심 구절마다 `<SPAN> ... </CN>` 또는 `<SPAN> ... </UN>` 태그를 붙입니다. 아래 표의 `P(</UN>)`가 기준값(0.01)보다 크면 다시 생성할 대상이 됩니다.

In [ ]:
df = phrase_confidence(rev_model, rev_proc, load_image(IMAGES["coco_cats"]), QUESTION)
df

## 4. 안저 영상에 적용하기

**데이터**: 공개 데이터만 쓰세요. 예를 들어 Kaggle [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection)에서 사진 몇 장을 받아 왼쪽 파일 패널에 올리면 됩니다. SKAIX 데이터는 이 노트북에 올리지 마세요.

**질문**: REVERSE는 일반 사진(COCO 등)으로만 학습했습니다. 의료 영상에서는
- 병변을 지어낼 때 `</UN>`으로 잘 표시하나요?
- 아니면 확신에 차서 틀린 말을 하나요?

이 질문이 연락 메일의 핵심 소재가 됩니다.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick 3-5 fundus images
FUNDUS = list(uploaded.keys())
FUNDUS

In [ ]:
FUNDUS_Q = ("Describe this retinal fundus image in detail. "
            "Are there any visible lesions such as hemorrhages, hard exudates, or microaneurysms?")

for path in FUNDUS:
    img = load_image(path)
    display(img.resize((256, 256)))
    ans, flagged, ok = reverse_generate(rev_model, rev_proc, img, FUNDUS_Q)
    conf = phrase_confidence(rev_model, rev_proc, img, FUNDUS_Q)
    results.append({"image": path, "model": "reverse", "answer": ans,
                    "flagged_phrases": "; ".join(flagged), "fully_corrected": ok})
    print(f"REVERSE: {ans}\nflagged: {flagged}")
    display(conf)

원래 모델과 비교하려면 2-1 셀처럼 기본 모델을 다시 불러와서 같은 사진에 `plain_generate`를 돌리면 됩니다. APTOS의 정답 등급(`train.csv`의 `diagnosis`, 0 = 정상 ~ 4 = 증식성)과 비교하면, 정상 사진에서 병변이 있다고 말하는지를 셀 수 있어요.

## 5. 결과 저장

In [ ]:
pd.DataFrame(results).to_csv("reverse_results.csv", index=False)
files.download("reverse_results.csv")

## 다음 단계

1. 정상 사진 10장, 병변 사진 10장 정도로 늘려서 "정상인데 병변이 있다고 말한 비율"을 원래 모델과 REVERSE로 비교하기
2. `un_threshold`를 0.003, 0.01, 0.03으로 바꿔가며 결과가 어떻게 달라지는지 보기
3. 결과를 그래프 1~2개와 함께 GitHub README로 정리하기